In [3]:
# =====================================================
# 1. Import thư viện & thiết lập cấu hình hiển thị
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Thiết lập style vẽ
plt.style.use("ggplot")

# Thiết lập hiển thị pandas
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Đã import xong các thư viện cần thiết.")

Đã import xong các thư viện cần thiết.


In [4]:
# =====================================================
# 2. Load dữ liệu từ file All_VN30_Merged (1).xlsx
# =====================================================

DATA_PATH = Path("All_VN30_Merged (1).xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {DATA_PATH}")

df = pd.read_excel(DATA_PATH)

print("Kích thước dữ liệu (rows, cols):", df.shape)
df.head()

Kích thước dữ liệu (rows, cols): (73153, 5)


,time,open,close,volume,symbol
0,2020-01-02,6.5800,6.6400,1163109,ACB
1,2020-01-02,6.5800,6.6400,1163109,ACB
2,2020-01-03,6.6400,6.6400,1055528,ACB
3,2020-01-03,6.6400,6.6400,1055528,ACB
4,2020-01-06,6.6400,6.4900,1286035,ACB


In [8]:
# =====================================================
# 3. Tổng quan cấu trúc & chất lượng dữ liệu
# =====================================================

print("=== Kiểu dữ liệu (dtypes) ===")
print(df.dtypes)

print("\n=== Thống kê missing values theo cột ===")
print(df.isna().sum())

# -----------------------------------------------------
# Loại bỏ cột 'time' khỏi thống kê mô tả (không có ý nghĩa mean/std)
# -----------------------------------------------------
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns]

print("\n=== Thống kê mô tả cơ bản cho các cột numeric ===")
display(df[numeric_cols].describe())

print("\n=== Thống kê mô tả cho các cột dạng object (ngoại trừ 'time') ===")
object_cols = [c for c in df.select_dtypes(include=["object"]).columns if c != "time"]
display(df[object_cols].describe())

# Thống kê số lượng symbol
symbols = df["symbol"].unique()
print("\nSố lượng symbol (cổ phiếu + chỉ số):", len(symbols))
print("Danh sách symbol:")
print(symbols)


=== Kiểu dữ liệu (dtypes) ===
time      datetime64[ns]
open             float64
close            float64
volume             int64
symbol            object
dtype: object

=== Thống kê missing values theo cột ===
time      0
open      0
close     0
volume    0
symbol    0
dtype: int64

=== Thống kê mô tả cơ bản cho các cột numeric ===


,open,close,volume
count,"73,153.0000","73,153.0000","73,153.0000"
mean,62.8237,62.8118,"10,124,336.7591"
std,175.9056,175.8531,"34,406,073.9119"
min,2.0500,2.0900,0.0000
25%,16.2100,16.1900,"1,025,521.0000"
50%,29.5000,29.5000,"2,702,900.0000"
75%,58.3100,58.2700,"7,577,812.0000"
max,"2,036.9000","2,022.2700","1,146,147,784.0000"



=== Thống kê mô tả cho các cột dạng object (ngoại trừ 'time') ===


,symbol
count,73153
unique,41
top,BID
freq,3000



Số lượng symbol (cổ phiếu + chỉ số): 41
Danh sách symbol:
['ACB' 'BCM' 'BID' 'BVH' 'CTG' 'DGC' 'EIB' 'FPT' 'GAS' 'GVR' 'HDB' 'HPG'
 'KDH' 'LPB' 'MBB' 'MSN' 'MWG' 'NVL' 'PDR' 'PLX' 'PNJ' 'REE' 'ROS' 'SAB'
 'SBT' 'SHB' 'SSB' 'SSI' 'STB' 'TCB' 'TCH' 'TPB' 'VCB' 'VHM' 'VIB' 'VIC'
 'VJC' 'VN30' 'VNM' 'VPB' 'VRE']


In [9]:
# =====================================================
# 4. Tiền xử lý dữ liệu & tạo biến mới
# =====================================================

# Đảm bảo cột thời gian đúng kiểu datetime
df["time"] = pd.to_datetime(df["time"])

# Sort dữ liệu theo symbol và time
df = df.sort_values(["symbol", "time"]).reset_index(drop=True)

# Xác định symbol đại diện cho chỉ số VN30
INDEX_SYMBOL = "VN30"
has_index = INDEX_SYMBOL in df["symbol"].unique()
print("Có symbol VN30 trong dữ liệu hay không?:", has_index)

if has_index:
    df_index = df[df["symbol"] == INDEX_SYMBOL].copy()
    df_stocks = df[df["symbol"] != INDEX_SYMBOL].copy()
else:
    df_index = pd.DataFrame()
    df_stocks = df.copy()

# Tạo biến lợi suất ngày (simple returns) cho từng symbol
df["returns"] = df.groupby("symbol")["close"].pct_change()

# Tạo biến log-returns (thường dùng trong tài chính)
df["log_returns"] = df.groupby("symbol")["close"].apply(
    lambda x: np.log(x) - np.log(x.shift(1))
)

print("Sau tiền xử lý:")
df.head()

Có symbol VN30 trong dữ liệu hay không?: True


TypeError: incompatible index of inserted column with frame index